# Temporal averaging of the coastal boundary

MODFLOW~6 integrates a whole coupling interval in one backward-Euler step, and that
step applies the boundary flux across the entire interval: the volume exchanged is
$\Delta t\,C\,(H-h)$. So the boundary head $H$ has to represent the interval, not the
instant at its end. Those coincide only while the boundary varies slowly, and a
tidal boundary sampled over an appreciable fraction of a tidal cycle does not.

Two reductions of the D-Flow FM stage and depth to that single boundary value are
compared, over coarse runs at four coupling intervals scored against a 30-minute
reference:

- **instant** -- the value at the end of the interval. Every scenario before
  August 2026 used this.
- **mean** -- a wetted-fraction weighted time average. For the GHB this is exact
  rather than approximate: the conductance is $C_0$ times a binary wet mask, so the
  interval-mean flux factors as $C_0\langle w\rangle(\langle w s_1\rangle/\langle
  w\rangle - h)$, making the wetted fraction the conductance multiplier and the
  stage a wet-weighted mean.

## Result

The two methods fail in opposite directions, and which one wins is decided by the
tide, not by the model:

- **instant** never damps amplitude, but aliases once the interval exceeds the
  Nyquist limit for the dominant constituent.
- **mean** never aliases, but damps amplitude by an amount that grows with the
  interval.

Below Nyquist the damping penalty exceeds the (absent) aliasing penalty and instant
is closer to the reference; above it, aliasing dominates and the mean wins by a
widening margin. The M2 period is 12.42 h, so the limit is 6.21 h -- and the tracer
error ratio crosses one between the 4 h and 8 h runs, which brackets it.

The rows where instant wins are immaterial in absolute terms: aquifer head RMSE is
under a millimetre at 2 and 4 h coupling for both methods. The interval where the
choice has physical consequence is daily, where instant carries a 33 mm head error
against 3 mm, and inflates peak tracer concentration by 62 %.

In [ ]:
%matplotlib inline
import pathlib as pl
import sys

import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
ROOT = pl.Path.cwd().parent
FIGS = ROOT / "docs" / "GP" / "figures"

# Which sweep to score. Both grids carry all seven coupling intervals under both
# reductions. Nothing else about the comparison is set here: the run names, the
# reference, and the statistics all live in the module loaded below, because a
# second copy of them here is exactly how this notebook came to disagree with it.
GRID = "coarse"          # "coarse" or "high"

# Intervals coarsest-last, for reading down the tables. The module keys on the same
# tags but sorts them alphabetically, which puts the daily run first.
ORDER = ["30.00M", "01.00H", "02.00H", "04.00H", "08.00H", "01.00D"]

In [ ]:
# The statistics and the figure each have one implementation, in
# docs/GP/scripts/. Duplicating them here would let the notebook and the manuscript
# drift apart, which is how the sequence figure once came to disagree with the text
# describing it.
sys.path.insert(0, str(ROOT / "docs" / "GP" / "scripts"))
import boundary_averaging_data as bad

ds, source = bad.load_or_refresh(grid=GRID)
print(f"{GRID}: " + ("recomputed from results/" if source == "results"
                     else f"read archive; {len(bad.missing(grid=GRID))} runs absent"))
stats = ds.to_dataframe().reset_index()
stats["hours"] = stats["interval"].map(dict(zip(ds.interval.values, ds.hours.values)))

# Taken from the archive rather than spelled out. Hard-coding the labels is what
# silently emptied these tables when the module moved its reference from the
# 30-minute run to the 15-minute one.
REFS = list(ds.ref.values)
REF0 = REFS[0]
PEAK_REF = float(ds.attrs["peak_reference_concentration"])
P9999_REF = float(ds.attrs["p9999_reference_concentration"])
P999_REF = float(ds.attrs["p999_reference_concentration"])
NYQUIST_H = float(ds.attrs["m2_nyquist_hours"])
print(f"references: {REFS}")
print(f"Nyquist limit for M2: {NYQUIST_H:.2f} h")
print(f"reference tracer  max {PEAK_REF:.4f}   99.99th {P9999_REF:.5f}   "
      f"99.9th {P999_REF:.5f}")

In [ ]:
for rlabel in REFS:
    s = stats[stats["ref"] == rlabel].set_index("interval").loc[ORDER]
    print(f"\n=== reference: {rlabel} ===")
    print("ratio = instant RMSE / mean RMSE;  > 1 means the mean is closer\n")
    print(s[["hours", "head_inst", "head_mean", "head_ratio",
             "seep_inst", "seep_mean", "seep_ratio",
             "trac_inst", "trac_mean", "trac_ratio"]]
          .rename(columns={"head_inst": "head_i(mm)", "head_mean": "head_m(mm)"})
          .round(4).to_string())

# peak is a max over the whole space-time field, and a max is the least robust
# statistic there is: on coarse it lands in cell 1669 in all twelve runs at 7.9x the
# 99.99th percentile, and on highres in one of two cells at 54x. So it is reported
# beside the percentiles, which no single cell can move. Where the two disagree the
# peak is describing a hot spot, not the domain.
s = stats[stats["ref"] == REF0].set_index("interval").loc[ORDER]
amp = pd.DataFrame({
    "peak i%": 100 * (s.peak_inst - PEAK_REF) / PEAK_REF,
    "peak m%": 100 * (s.peak_mean - PEAK_REF) / PEAK_REF,
    "p99.99 i%": 100 * (s.p9999_inst - P9999_REF) / P9999_REF,
    "p99.99 m%": 100 * (s.p9999_mean - P9999_REF) / P9999_REF,
    "p99.9 i%": 100 * (s.p999_inst - P999_REF) / P999_REF,
    "p99.9 m%": 100 * (s.p999_mean - P999_REF) / P999_REF,
})
print(f"\n\ntracer amplitude, percent departure from the {REF0} reference")
print(amp.round(1).to_string())

In [ ]:
# The figure is drawn by docs/GP/scripts/make_boundary_averaging_figure.py from the
# archive that load_or_refresh wrote above, not from the simulation output: that
# output is tens of gigabytes and is not in version control, so a co-author with only
# the repository could not otherwise rebuild a manuscript figure. This cell used to
# rebuild the summary and write a second archive of its own, to a different path than
# the module's -- which is the drift the single implementation exists to prevent.
import make_boundary_averaging_figure as mbaf

mbaf.make(GRID)
print("figure:", mbaf.output_path(GRID))

In [ ]:
# Kept for interactive use; the manuscript figure is the one written above. Panel D
# carries the 99.99th percentile rather than the max, so this view shows the
# field-wide amplitude where the manuscript figure shows the hot-spot one.
C_I, C_M = "#d62728", "#1f77b4"
s = stats[stats["ref"] == REF0].sort_values("hours")
h = s["hours"].to_numpy()
# Ticks follow the archive, so adding an interval does not silently drop a point.
TICK_LAB = [f"{v * 60:.0f} min" if v < 1 else f"{v:.0f} h" if v < 24
            else f"{v / 24:.0f} d" for v in h]

with styles.USGSPlot():
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(7.5, 5.2), layout="constrained")
    panels = [(axs[0, 0], "head_inst", "head_mean", "Aquifer head RMSE, in millimeters", True),
              (axs[0, 1], "seep_inst", "seep_mean", "Sewer seepage RMSE, in cubic feet per day", True),
              (axs[1, 0], "trac_inst", "trac_mean", "Sewer tracer RMSE, dimensionless", True)]
    for i, (ax, ci, cm, lab, logy) in enumerate(panels):
        ax.plot(h, s[ci], "o-", color=C_I, lw=1.2, ms=4, label="instantaneous")
        ax.plot(h, s[cm], "s-", color=C_M, lw=1.2, ms=4, label="time-averaged")
        ax.set_xscale("log")
        if logy:
            ax.set_yscale("log")
        ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
        ax.set_xticks(h)
        ax.set_xticklabels(TICK_LAB, fontsize=7)
        ax.tick_params(labelsize=7, top=False)
        styles.heading(ax=ax, letter="ABCD"[i], heading=lab, fontsize=7.5)

    ax = axs[1, 1]
    ax.axhline(P9999_REF, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.plot(h, s["p9999_inst"], "o-", color=C_I, lw=1.2, ms=4)
    ax.plot(h, s["p9999_mean"], "s-", color=C_M, lw=1.2, ms=4)
    ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.set_xscale("log")
    ax.set_xticks(h)
    ax.set_xticklabels(TICK_LAB, fontsize=7)
    ax.tick_params(labelsize=7, top=False)
    styles.heading(ax=ax, letter="D",
                   heading="Sewer tracer, 99.99th percentile", fontsize=7.5)
    ax.annotate("reference", xy=(h.min(), P9999_REF), xytext=(0, 3),
                textcoords="offset points", fontsize=6.5, color="0.35")

    for ax in axs.flat:
        styles.xlabel(ax=ax, label="Coupling interval")
        ax.annotate(r"$M_2$ Nyquist", xy=(NYQUIST_H, 0.96),
                    xycoords=("data", "axes fraction"),
                    xytext=(3, 0), textcoords="offset points",
                    fontsize=6.5, color="0.35", va="top", ha="left")
    hs, ls = axs[0, 0].get_legend_handles_labels()
    styles.graph_legend(ax=axs[1, 0], handles=hs, labels=ls, loc="lower center",
                        bbox_to_anchor=(1.05, -0.42), ncol=2, frameon=False, fontsize=7.5)

### Reading the figure

Panels A--C are RMSE against the 30-minute reference; D is the peak tracer
concentration, with the reference value dashed. The vertical rule is the Nyquist
limit for M2, 6.21 h.

Below the limit the two curves are close and both are small -- aquifer head RMSE is
under a millimetre at 2 and 4 h -- so the choice there is immaterial even where the
ratio favours instantaneous sampling. Above it the curves separate: instantaneous
peak tracer concentration leaves the reference abruptly, reaching +27 % at 8 h and
+62 % at daily coupling, while the averaged boundary stays within a couple of
percent until daily.

That abruptness is the signature. A first-order truncation error grows smoothly with
the step; aliasing does not appear at all until the sampling interval crosses the
limit, and then grows quickly. The two methods are not better and worse versions of
the same approximation -- they fail by different mechanisms, and the tide decides
which mechanism is active.